# Marketplace Causal Inference

## Business question

Does sponsored advertising create **incremental** bookings in a marketplace when suppliers choose whether and when to participate?

This notebook walks through the causal reasoning, diagnostics, and business interpretation. The full reusable implementation lives in `src/`.

In [ ]:
from src.generate_data import generate_marketplace_panel
import pandas as pd
import numpy as np

df = generate_marketplace_panel(n_properties=1000, n_weeks=52, seed=42)
df.head()

## 1. Why treated vs. untreated is not causal

Adoption is voluntary and depends partly on recent demand growth and supplier characteristics. Therefore, treated suppliers are systematically different from untreated suppliers.

In [ ]:
df.groupby('treated')['bookings'].agg(['mean','median','count'])

## 2. Two-way fixed effects

We control for time-invariant supplier differences and common weekly shocks. This is a useful baseline, but staggered treatment timing motivates dynamic event-study diagnostics.

In [ ]:
import statsmodels.formula.api as smf

d = df.copy()
d['log_bookings'] = np.log1p(d['bookings'])
twfe = smf.ols('log_bookings ~ treated + C(property_id) + C(week)', data=d).fit(
    cov_type='cluster', cov_kwds={'groups': d['property_id']}
)
twfe.params['treated'], (np.exp(twfe.params['treated']) - 1) * 100

## 3. Event-study logic

A credible design should show limited evidence of differential pre-trends before adoption and a plausible post-treatment response afterward. The production script in `src/analysis.py` creates event-time indicators and exports confidence intervals.

## 4. Heterogeneous treatment effects

The simulation is intentionally designed so that advertising is more valuable for suppliers with lower baseline visibility. This mirrors a realistic marketplace question: **should the program be broadly offered, or targeted to suppliers for whom incremental exposure is most valuable?**

## 5. Business recommendation

The correct decision should combine statistical evidence with economic magnitude. A positive average treatment effect alone is insufficient if the gain is concentrated in a subset of suppliers or if broad adoption reduces program efficiency.

A strong recommendation therefore focuses on **incrementality, heterogeneity, and targeting** rather than raw treated-vs-untreated differences.